# Practice Lab: Design of Machine Learning Experiments
In this lab, you will explore techniques to evaluate and improve your machine learning models.

# Outline
- [ 1 - Setup & Packages ](#1)
- [ 2 - Evaluating a Learning Algorithm (Polynomial Regression)](#2)
  - [ 2.1 Splitting your data set](#2.1)
  - [ 2.2 Error calculation for model evaluation, linear regression](#2.2)
    - [ Exercise 1](#ex01)
  - [ 2.3 Compare performance on training and test data](#2.3)
    - [ Exercise 2](#ex02)
- [ 3 - Bias and Variance](#3)
  - [ 3.1 Plot Train, Cross-Validation](#3.1)
  - [ 3.2 Finding the optimal degree](#3.2)
     - [ Exercise 3](#ex03)
  - [ 3.3 Tuning Regularization.](#3.3)
     - [ Exercise 4](#ex04)
  - [ 3.4 Getting more data: Increasing Training Set Size (m)](#3.4)
     - [ Exercise 5](#ex05)
- [ 4 - Evaluating a Learning Algorithm (Neural Network)](#4)
  - [ 4.1 Data Set](#4.1)
  - [ 4.2 Evaluating categorical model by calculating classification error](#4.2)
    - [ Exercise 6](#ex06)
  - [ 4.3 - Model Complexity](#4.3)
    - [ Exercise 7](#ex07)
    - [ Exercise 8](#ex08)
  - [ 4.4 - Regularization](#4.4)
    - [ Exercise 9](#ex09)
  - [ 4.5 Test](#4.5)


<a name="1"></a>
# 1 -  Setup and Packages



Let's run the cell below to import all the packages that you will need during this assignment.
- [numpy](https://numpy.org/) is the fundamental package for scientific computing Python.
- [matplotlib](http://matplotlib.org) is a popular library to plot graphs in Python.
- [scikitlearn](https://scikit-learn.org/stable/) is a basic library for data mining
- [tensorflow](https://www.tensorflow.org/) a popular platform for machine learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import relu,linear
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

<a name="2"></a>
## 2 - Evaluating a Learning Algorithm (Polynomial Regression)

Let's say you have created a machine learning model and you find it *fits* your training data very well. You're done? Not quite. The goal of creating the model was to be able to predict values for <span style="color:blue">*new* </span> examples.

How can you test your model's performance on new data before deploying it?   
The answer has two parts:
* Split your original data set into "Training" and "Test" sets.
    * Use the training data to fit the parameters of the model
    * Use the test data to evaluate the model on *new* data
* Develop an error function to evaluate your model.

<a name="2.1"></a>
### 2.1 Splitting your data set
Lectures advised reserving 20-40% of your data set for testing. Let's use an `sklearn` function [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to perform the split. Double-check the shapes after running the following cell.

In [ ]:
def gen_data(m, seed=1, scale=0.7):
    """ generate a data set based on a x^2 with added noise """
    c = 0
    x_train = np.linspace(0,49,m)
    np.random.seed(seed)
    y_ideal = x_train**2 + c
    y_train = y_ideal + scale * y_ideal*(np.random.sample((m,))-0.5)
    x_ideal = x_train #for redraw when new data included in X
    return x_train, y_train, x_ideal, y_ideal

In [ ]:
# Generate some data
X,y,x_ideal,y_ideal = gen_data(18, 2, 0.7)
print("X.shape", X.shape, "y.shape", y.shape)

#split the data using sklearn routine
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33, random_state=1)
print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_test.shape", X_test.shape, "y_test.shape", y_test.shape)

#### 2.1.1 Plot Train, Test sets
You can see below the data points that will be part of training (in red) are intermixed with those that the model is not trained on (test). This particular data set is a quadratic function with noise added. The "ideal" curve is shown for reference.

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(x_ideal, y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
ax.set_title("Training, Test",fontsize = 14)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(X_train, y_train, color = "red",           label="train")
ax.scatter(X_test, y_test,   color = "blue",   label="test")
ax.legend(loc='upper left')
plt.show()

<a name="2.2"></a>
### 2.2 Error calculation for model evaluation, linear regression
When *evaluating* a linear regression model, you average the squared error difference of the predicted values and the target values.

$$ J_\text{test}(\mathbf{w},b) =
            \frac{1}{2m_\text{test}}\sum_{i=0}^{m_\text{test}-1} ( f_{\mathbf{w},b}(\mathbf{x}^{(i)}_\text{test}) - y^{(i)}_\text{test} )^2
            \tag{1}
$$

<a name="ex01"></a>
### Exercise 1

Below, create a function to evaluate the error on a data set for a linear regression model.

In [ ]:
def eval_mse(y, yhat):
    """
    Calculate the mean squared error on a data set.
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:
      err: (scalar)
    """
    ### START CODE HERE ###


    ### END CODE HERE ###

In [ ]:
y_hat = np.array([2.4, 4.2])
y_tmp = np.array([2.3, 4.1])
print("err =",eval_mse(y_hat, y_tmp))

<details>
  <summary><font size="3" color="darkblue"><b>Expected Output (Click to expand)</b></font></summary>
err = 0.0050000000000000305

<a name="2.3"></a>
### 2.3 Compare performance on training and test data
Let's build a high degree polynomial model to minimize training error. This will use the linear_regression functions from `sklearn`.

<a name="ex02"></a>
### Exercise 2
Build a polynomial regression model of degree 10, apply feature scaling, and evaluate its performance on training and test data using Mean Squared Error (MSE).
1.   Transform X_train and X_test into polynomial features of degree 10 using
PolynomialFeatures.
2.   Standardize the polynomial features using StandardScaler.
3.   Train a linear regression model on the scaled training data.
4.   Compute the MSE for both training and test predictions using the eval_mse function.

In [ ]:
### START CODE HERE ###


### END CODE HERE ###

The computed error on the training set is substantially less than that of the test set.

In [ ]:
print(f"training err {train_error:0.2f}, test err {test_error:0.2f}")

<details>
  <summary><font size="3" color="darkblue"><b>Expected Output (Click to expand)</b></font></summary>
training err 58.01, test err 171215.01

## What can you conclude ?

<a name="3"></a>
## 3 - Bias and Variance
The test set error shows this model will not work well on new data. If you use the test error to guide improvements in the model, then the model will perform well on the test data... but the test data was meant to represent *new* data.
You need yet another set of data to test new data performance.

It is clear that the degree of the polynomial model is too high. How can you choose a good value? It turns out that the training and cross-validation performance can provide guidance. By trying a range of degree values, the training and cross-validation performance can be evaluated. As the degree becomes too large, the cross-validation performance will start to degrade relative to the training performance. Let's try this on our example.

The distribution of training, cross-validation and test sets shown in the below table is a typical distribution, but can be varied depending on the amount of data available.

| data             | % of total | Description |
|------------------|:----------:|:---------|
| training         | 60         | Data used to tune model parameters $w$ and $b$ in training or fitting |
| cross-validation | 20         | Data used to tune other model parameters like degree of polynomial, regularization or the architecture of a neural network.|
| test             | 20         | Data used to test the model after tuning to gauge performance on new data |


Let's create three datasets: we'll use train_test_split from sklearn once to split the data into a training set and a cross-validation (CV) set, and the test set with 100 samples is provided separately.

In [ ]:
# Generate  data
X,y, x_ideal,y_ideal = gen_data(40, 5, 0.7)

X_test = np.linspace(0,int(X.max()),100)
print("X.shape", X.shape, "y.shape", y.shape)

#split the data using sklearn routine
X_train, X_cv, y_train, y_cv = train_test_split(X,y,test_size=0.40, random_state=1)
print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_cv.shape", X_cv.shape, "y_cv.shape", y_cv.shape)

<a name="3.1"></a>
### 3.1 Plot Train, Cross-Validation
You can see below the datapoints that will be part of training (in red) are intermixed with those that the model is not trained on (cv).

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(x_ideal, y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
ax.set_title("Training, CV",fontsize = 14)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(X_train, y_train, color = "red",           label="train")
ax.scatter(X_cv, y_cv,       color = "orange", label="cv")
ax.legend(loc='upper left')
plt.show()

<a name="3.2"></a>
### 3.2 Finding the optimal degree
In previous labs (lab 3), you found that you could create a model capable of fitting complex curves by utilizing a polynomial.  Further, you demonstrated that by increasing the *degree* of the polynomial, you could *create* overfitting. Let's use that knowledge here to test our ability to tell the difference between over-fitting and under-fitting.

<a name="ex03"></a>
### Exercise 3
You are given a dataset with training features X_train, training labels y_train, cross-validation features X_cv, and cross-validation labels y_cv. Your task is to:

1. Train polynomial regression models with degrees ranging from 1 to 9. Remember to:
 * Transform the training and cross-validation data into polynomial features.
 * Standardize the features using StandardScaler.

2. Evaluate the performance of each model using the training and cross-validation datasets. Store the training error in err_train and the cross-validation error in err_cv.
3. Identify the optimal polynomial degree that minimizes the cross-validation error.

In [ ]:
max_degree = 9
err_train = np.zeros(max_degree)
err_cv = np.zeros(max_degree)

y_pred = np.zeros((100,max_degree))  #columns are lines to plot

scalar = StandardScaler()

for degree in range(max_degree):
    # CODE STARTS HERE


    # CODE ENDS HERE

    x_poly = poly.transform(X_test.reshape(-1,1))
    x_poly = scalar.transform(x_poly)
    y_pred[:,degree] = model.predict(x_poly)

optimal_degree = np.argmin(err_cv)+1

<font size="4">Let's plot the result:</font>

In [ ]:
def plt_optimal_degree(X_train, y_train, X_cv, y_cv, x, y_pred, x_ideal, y_ideal, err_train, err_cv, optimal_degree, max_degree):
    # Plot predictions vs data for each degree in separate figures
    for i in range(max_degree):
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        fig.canvas.toolbar_visible = False
        fig.canvas.header_visible = False
        fig.canvas.footer_visible = False

        # Plot predictions vs data
        ax[0].set_title(f"Predictions vs Data (Degree {i+1})", fontsize=12)
        ax[0].set_xlabel("x")
        ax[0].set_ylabel("y")

        ax[0].plot(x_ideal, y_ideal, "--", color="orangered", label="y_ideal", lw=1)
        ax[0].scatter(X_train, y_train, color="red", label="train")
        ax[0].scatter(X_cv, y_cv, color="orange", label="cv")
        ax[0].plot(x, y_pred[:, i], lw=1, label=f"Degree {i+1}")
        ax[0].legend(loc='upper left')

        # Plot error vs degree
        ax[1].set_title("Error vs Degree", fontsize=12)
        cpts = list(range(1, max_degree + 1))
        ax[1].plot(cpts, err_train[0:], marker='o', label="train error", lw=2, color="blue")
        ax[1].plot(cpts, err_cv[0:], marker='o', label="cv error", lw=2, color="orange")
        ax[1].axvline(x=i+1, color="red", linestyle="--", lw=1, label=f"Degree {i+1}")
        ax[1].set_xlabel("degree")
        ax[1].set_ylabel("error")
        ax[1].legend()

        fig.suptitle(f"Degree {i+1}: Predictions and Errors", fontsize=12)
        plt.tight_layout()
        plt.show()

    fig.suptitle("Find Optimal Degree", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.close("all")
plt_optimal_degree(X_train, y_train, X_cv, y_cv, X_test, y_pred, x_ideal, y_ideal,
                   err_train, err_cv, optimal_degree, max_degree)

## Questions:

1. If you were to deploy this model, which degree would you choose and why?
2. How do the training and cross-validation errors behave as the degree increases? What does this tell you about variance and bias?

<a name="3.3"></a>
### 3.3 Tuning Regularization.
In previous labs, you have utilized *regularization* to reduce overfitting. Similar to degree, one can use the same methodology to tune the regularization parameter lambda ($\lambda$).

<a name="ex04"></a>
### Exercise 4
Your task is to train polynomial regression models with different regularization strengths (lambda) and evaluate their performance using training and cross-validation datasets. You will:

1. Train models with varying regularization parameters. Remember to:
 * Transform the training and cross-validation data into polynomial features.
 * Standardize the features using StandardScaler.
 * Train a regularized linear regression model (Ridge Regression) on the training data.

2. Evaluate the performance of each model using the training and cross-validation datasets. Store the training error in err_train and the cross-validation error in err_cv.

3. Identify the optimal regularization parameter that minimizes cross-validation error.


In [ ]:
lambda_range = np.array([0.0, 1e-6, 1e-5, 1e-4,1e-3,1e-2, 1e-1,1,10,100,1000])
num_steps = len(lambda_range)
degree = 10
err_train = np.zeros(num_steps)
err_cv = np.zeros(num_steps)

y_pred = np.zeros((100,num_steps))  #columns are lines to plot

for i in range(num_steps):
    lambda_= lambda_range[i]

    # CODE STARTS HERE

    # CODE ENDS HERE

    x_poly = poly.transform(X_test.reshape(-1,1))
    x_poly = scalar.transform(x_poly)
    y_pred[:,i] = model.predict(x_poly)

optimal_reg_idx = np.argmin(err_cv)

In [ ]:
def plt_tune_regularization(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, optimal_reg_idx, lambda_range):
    # Plot each regularization curve in a separate figure
    for i, reg_lambda in enumerate(lambda_range):
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        fig.canvas.toolbar_visible = False
        fig.canvas.header_visible = False
        fig.canvas.footer_visible = False

        # Plot predictions vs data
        ax[0].set_title(f"Predictions vs Data ($\lambda = {reg_lambda}$)", fontsize=12)
        ax[0].set_xlabel("x")
        ax[0].set_ylabel("y")

        ax[0].scatter(X_train, y_train, color="red", label="train")
        ax[0].scatter(X_cv, y_cv, color="orange", label="cv")
        ax[0].plot(x, y_pred[:, i], lw=1, label=f"$\lambda = {reg_lambda}$")
        ax[0].legend()

        # Plot error vs regularization
        ax[1].set_title("Error vs Regularization", fontsize=12)
        ax[1].plot(lambda_range, err_train, label="train error", color="blue")
        ax[1].plot(lambda_range, err_cv, label="cv error", color="orange")
        ax[1].set_xscale('log')
        ax[1].axvline(x=reg_lambda, color="red", linestyle="--", lw=1, label=f"Current $\lambda = {reg_lambda}$")
        ax[1].set_xlabel("Regularization ($\lambda$)")
        ax[1].set_ylabel("Error")
        ax[1].legend()

        fig.suptitle(f"Regularization Analysis ($\lambda = {reg_lambda}$)", fontsize=12)
        plt.tight_layout()
        plt.show()



    fig.suptitle("Tuning Regularization", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.close("all")
plt_tune_regularization(X_train, y_train, X_cv, y_cv, X_test, y_pred, err_train, err_cv, optimal_reg_idx, lambda_range)

## Questions
1. What is the optimal value of lambda? Why is this value considered optimal?
2. At what value of lambda does the model start to underfit? How can you tell from the plot?

<a name="3.4"></a>
### 3.4 Getting more data: Increasing Training Set Size (m)
When a model is overfitting (high variance), collecting additional data can improve performance. Let's try that here.

In [ ]:
m = 50
m_range = np.array(m*np.arange(1,16))
num_steps = m_range.shape[0]
degree = 16
err_train = np.zeros(num_steps)
err_cv = np.zeros(num_steps)
y_pred = np.zeros((100,num_steps))

for i in range(num_steps):
        X, y, y_ideal, x_ideal = gen_data(m_range[i],5,0.7)
        x = np.linspace(0,int(X.max()),100)
        X_train, X_cv, y_train, y_cv = train_test_split(X,y,test_size=0.40, random_state=1)

        # CODE STARTS HERE

        # CODE ENDS HERE

        x_poly = poly.transform(X_test.reshape(-1,1))
        x_poly = scalar.transform(x_poly)
        y_pred[:,i] = model.predict(x_poly)


In [ ]:
def plt_tune_m(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, m_range, degree):
    # Plot each m in a separate figure
    for i, m in enumerate(m_range):
        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        fig.canvas.toolbar_visible = False
        fig.canvas.header_visible = False
        fig.canvas.footer_visible = False

        # Plot predictions vs data
        ax[0].set_title(f"Predictions vs Data ($m = {m}$)", fontsize=12)
        ax[0].set_xlabel("x")
        ax[0].set_ylabel("y")

        ax[0].scatter(X_train, y_train, color="red", s=3, label="train", alpha=0.4)
        ax[0].scatter(X_cv, y_cv, color="orange", s=3, label="cv", alpha=0.4)
        ax[0].plot(x, y_pred[:, i], lw=1, label=f"$m = {m}$")
        ax[0].legend(loc='upper left')
        ax[0].text(0.05, 0.5, f"degree = {degree}", fontsize=10, ha='left', transform=ax[0].transAxes, color="blue")

        # Plot error vs number of examples
        ax[1].set_title("Error vs Number of Examples", fontsize=12)
        ax[1].plot(m_range, err_train, label="train error", color="blue")
        ax[1].plot(m_range, err_cv, label="cv error", color="orange")
        ax[1].axvline(x=m, color="red", linestyle="--", lw=1, label=f"Current $m = {m}$")
        ax[1].set_xlabel("Number of Examples (m)")
        ax[1].set_ylabel("Error")
        ax[1].legend()

        fig.suptitle(f"Tuning Number of Examples ($m = {m}$)", fontsize=12)
        plt.tight_layout()
        plt.show()


    fig.suptitle("Tuning Number of Examples", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
plt_tune_m(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, m_range, degree)

<a name="ex05"></a>
### Exercise 5

Discuss the above plots.

<a name="4"></a>
## 4 - Evaluating a Learning Algorithm (Neural Network)
Above, you tuned aspects of a polynomial regression model. Here, you will work with a neural network model. Let's start by creating a classification data set.

<a name="4.1"></a>
### 4.1 Data Set
Run the cell below to generate a data set and split it into training, cross-validation (CV) and test sets. In this example, we're increasing the percentage of cross-validation data points for emphasis.  

In [ ]:
from sklearn.datasets import make_blobs
def gen_blobs():
    classes = 6
    m = 800
    std = 0.4
    centers = np.array([[-1, 0], [1, 0], [0, 1], [0, -1],  [-2,1],[-2,-1]])
    X, y = make_blobs(n_samples=m, centers=centers, cluster_std=std, random_state=2, n_features=2)
    return (X, y, centers, classes, std)

In [ ]:
# Generate and split data set
X, y, centers, classes, std = gen_blobs()

# split the data. Large CV population for demonstration
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.50, random_state=1)
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.20, random_state=1)
print("X_train.shape:", X_train.shape, "X_cv.shape:", X_cv.shape, "X_test.shape:", X_test.shape)

<a name="4.2"></a>
### 4.2 Evaluating categorical model by calculating classification error
The evaluation function for categorical models used here is simply the fraction of incorrect predictions:  
$$ J_{cv} =\frac{1}{m}\sum_{i=0}^{m-1}
\begin{cases}
    1, & \text{if $\hat{y}^{(i)} \neq y^{(i)}$}\\
    0, & \text{otherwise}
\end{cases}
$$

<a name="ex06"></a>
### Exercise 6

Below, complete the routine to calculate classification error. Note, in this lab, target values are the index of the category and are not [one-hot encoded](https://en.wikipedia.org/wiki/One-hot).

In [ ]:
def eval_cat_err(y, yhat):
    """
    Calculate the categorization error
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:|
      cerr: (scalar)
    """
    ### START CODE HERE ###

    ### END CODE HERE ###


In [ ]:
y_hat = np.array([1, 2, 0])
y_tmp = np.array([1, 2, 3])
print(f"categorization error {np.squeeze(eval_cat_err(y_hat, y_tmp)):0.3f}, expected:0.333" )
y_hat = np.array([[1], [2], [0], [3]])
y_tmp = np.array([[1], [2], [1], [3]])
print(f"categorization error {np.squeeze(eval_cat_err(y_hat, y_tmp)):0.3f}, expected:0.250" )

<a name="4.3"></a>
## 4.3 - Model Complexity
Below, you will build two models. A complex model and a simple model. You will evaluate the models to determine if they are likely to overfit or underfit.

###  Complex model

<a name="ex07"></a>
### Exercise 7
Below, compose a three-layer model:
* Dense layer with 120 units, relu activation
* Dense layer with 40 units, relu activation
* Dense layer with 6 units and a linear activation (not softmax)  

Compile using
* loss with `SparseCategoricalCrossentropy`, remember to use  `from_logits=True`
* Adam optimizer with learning rate of 0.01.

In [ ]:
tf.random.set_seed(1234)
model = Sequential(
    [
        ### START CODE HERE ###

        ### END CODE HERE ###
    ], name="Complex"
)
model.compile(
    ### START CODE HERE ###

    ### END CODE HERE ###
)

In [ ]:
model.fit(
    X_train, y_train,
    epochs=1000
)

In [ ]:
model.summary()

<details>
  <summary><font size="3" color="darkblue"><b>Expected Output (Click to expand)</b></font></summary>
Summary should match this (layer instance names may increment )
    
```
Model: "Complex"
_________________________________________________________________
Layer (type)                 Output Shape              Param #
=================================================================
L1 (Dense)                   (None, 120)               360
_________________________________________________________________
L2 (Dense)                   (None, 40)                4840
_________________________________________________________________
L3 (Dense)                   (None, 6)                 246
=================================================================
Total params: 5,446
Trainable params: 5,446
Non-trainable params: 0
_________________________________________________________________
```


In [ ]:
import matplotlib as mpl

dkcolors = plt.cm.Paired((1,3,7,9,5,11))
ltcolors = plt.cm.Paired((0,2,6,8,4,10))
dkcolors_map = mpl.colors.ListedColormap(dkcolors)
ltcolors_map = mpl.colors.ListedColormap(ltcolors)

def plt_mc_data(ax, X, y, classes,  class_labels=None, map=plt.cm.Paired, legend=False,size=50, m='o'):
    for i in range(classes):
        idx = np.where(y == i)
        col = len(idx[0])*[i]
        label = class_labels[i] if class_labels else "c{}".format(i)
        ax.scatter(X[idx, 0], X[idx, 1],  marker=m,
                    c=col, vmin=0, vmax=map.N, cmap=map,
                    s=size, label=label)
    if legend: ax.legend()
    ax.axis('equal')

def plot_cat_decision_boundary(ax, X,predict , class_labels=None, legend=False, vector=True, color='g', lw = 1):

    # create a mesh to points to plot
    pad = 0.5
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    h = max(x_max-x_min, y_max-y_min)/200
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    points = np.c_[xx.ravel(), yy.ravel()]
    #print("points", points.shape)
    #make predictions for each point in mesh
    if vector:
        Z = predict(points)
    else:
        Z = np.zeros((len(points),))
        for i in range(len(points)):
            Z[i] = predict(points[i].reshape(1,2))
    Z = Z.reshape(xx.shape)

    #contour plot highlights boundaries between values - classes in this case
    ax.contour(xx, yy, Z, colors=color, linewidths=lw)
    ax.axis('tight')

def plt_nn(model_predict,X_train,y_train, classes, X_cv, y_cv, suptitle=""):
    #plot the decison boundary.
    fig,ax = plt.subplots(1,2, figsize=(8,4))
    fig.canvas.toolbar_visible = False
    fig.canvas.header_visible = False
    fig.canvas.footer_visible = False
    plot_cat_decision_boundary(ax[0], X_train, model_predict,  vector=True)
    ax[0].set_title("training data", fontsize=14)

    #add the original data to the decison boundary
    plt_mc_data(ax[0], X_train,y_train, classes, map=dkcolors_map, legend=True, size=75)
    ax[0].set_xlabel('x0') ; ax[0].set_ylabel("x1");

    plot_cat_decision_boundary(ax[1], X_train, model_predict,  vector=True)
    ax[1].set_title("cross-validation data", fontsize=14)
    plt_mc_data(ax[1], X_cv,y_cv, classes,
                map=ltcolors_map, legend=True, size=100, m='<')
    ax[1].set_xlabel('x0') ; ax[1].set_ylabel("x1");
    fig.suptitle(suptitle,fontsize = 12)
    plt.show()

In [ ]:
#make a model for plotting routines to call
model_predict = lambda Xl: np.argmax(tf.nn.softmax(model.predict(Xl)).numpy(),axis=1)
plt_nn(model_predict,X_train,y_train, classes, X_cv, y_cv, suptitle="Complex Model")

This model has worked very hard to capture outliers of each category. As a result, it has miscategorized some of the cross-validation data. Let's calculate the classification error.

In [ ]:
training_cerr_complex = eval_cat_err(y_train, model_predict(X_train))
cv_cerr_complex = eval_cat_err(y_cv, model_predict(X_cv))
print(f"categorization error, training, complex model: {training_cerr_complex:0.3f}")
print(f"categorization error, cv,       complex model: {cv_cerr_complex:0.3f}")

### Simple model
Now, let's try a simple model

<a name="ex08"></a>
### Exercise 8

Below, compose a two-layer model:
* Dense layer with 6 units, relu activation
* Dense layer with 6 units and a linear activation.

Compile using
* loss with `SparseCategoricalCrossentropy`, remember to use  `from_logits=True`
* Adam optimizer with learning rate of 0.01.

In [ ]:
tf.random.set_seed(1234)
model_s = Sequential(
    [
        ### START CODE HERE ###

        ### END CODE HERE ###
    ], name = "Simple"
)
model_s.compile(
    ### START CODE HERE ###

    ### START CODE HERE ###
)


In [ ]:
model_s.summary()

<details>
  <summary><font size="3" color="darkblue"><b>Expected Output (Click to expand)</b></font></summary>
Summary should match this (layer instance names may increment )
    
```
Model: "Simple"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                      │ (None, 6)                   │              18 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 6)                   │              42 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘
 Total params: 182 (732.00 B)
 Trainable params: 60 (240.00 B)
 Non-trainable params: 0 (0.00 B)
 Optimizer params: 122 (492.00 B)
```

In [ ]:
model_s.fit(
    X_train,y_train,
    epochs=1000
)

In [ ]:
#make a model for plotting routines to call
model_predict_s = lambda Xl: np.argmax(tf.nn.softmax(model_s.predict(Xl)).numpy(),axis=1)
plt_nn(model_predict_s,X_train,y_train, classes, X_cv, y_cv, suptitle="Simple Model")

This simple models does pretty well. Let's calculate the classification error.

In [ ]:
training_cerr_simple = eval_cat_err(y_train, model_predict_s(X_train))
cv_cerr_simple = eval_cat_err(y_cv, model_predict_s(X_cv))
print(f"categorization error, training, simple model, {training_cerr_simple:0.3f}, complex model: {training_cerr_complex:0.3f}" )
print(f"categorization error, cv,       simple model, {cv_cerr_simple:0.3f}, complex model: {cv_cerr_complex:0.3f}" )

Our simple model has a little higher classification error on training data but does better on cross-validation data than the more complex model.

<a name="4.4"></a>
## 4.4 - Regularization
As in the case of polynomial regression, one can apply regularization to moderate the impact of a more complex model. Let's try this below.

<a name="ex09"></a>
### Exercise 9

Reconstruct your complex model, but this time include regularization.
Below, compose a three-layer model:
* Dense layer with 120 units, relu activation, `kernel_regularizer=tf.keras.regularizers.l2(0.1)`
* Dense layer with 40 units, relu activation, `kernel_regularizer=tf.keras.regularizers.l2(0.1)`
* Dense layer with 6 units and a linear activation.

Compile using
* loss with `SparseCategoricalCrossentropy`, remember to use  `from_logits=True`
* Adam optimizer with learning rate of 0.01.

In [ ]:
tf.random.set_seed(1234)
model_r = Sequential(
    [
        ### START CODE HERE ###

        ### START CODE HERE ###
    ], name= None
)
model_r.compile(
    ### START CODE HERE ###

    ### START CODE HERE ###
)


In [ ]:
model_r.summary()

<details>
  <summary><font size="3" color="darkblue"><b>Expected Output (Click to expand)</b></font></summary>
Summary should match this (layer instance names may increment )
    
```
Model: "sequential_1"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                     │ (None, 120)                 │             360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 40)                  │           4,840 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ (None, 6)                   │             246 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘
 Total params: 16,340 (63.83 KB)
 Trainable params: 5,446 (21.27 KB)
 Non-trainable params: 0 (0.00 B)
 Optimizer params: 10,894 (42.56 KB)
```


In [ ]:
model_r.fit(
    X_train, y_train,
    epochs=1000
)

In [ ]:
#make a model for plotting routines to call
model_predict_r = lambda Xl: np.argmax(tf.nn.softmax(model_r.predict(Xl)).numpy(),axis=1)

plt_nn(model_predict_r, X_train,y_train, classes, X_cv, y_cv, suptitle="Regularized")

The results look very similar to the 'ideal' model. Let's check classification error.

In [ ]:
training_cerr_reg = eval_cat_err(y_train, model_predict_r(X_train))
cv_cerr_reg = eval_cat_err(y_cv, model_predict_r(X_cv))
test_cerr_reg = eval_cat_err(y_test, model_predict_r(X_test))
print(f"categorization error, training, regularized: {training_cerr_reg:0.3f}, simple model, {training_cerr_simple:0.3f}, complex model: {training_cerr_complex:0.3f}" )
print(f"categorization error, cv,       regularized: {cv_cerr_reg:0.3f}, simple model, {cv_cerr_simple:0.3f}, complex model: {cv_cerr_complex:0.3f}" )

##Questions

What do you observe?

## Iterate to find optimal regularization value
As you did in linear regression, you can try many regularization values. This code takes several minutes to run.

In [ ]:
tf.random.set_seed(1234)
lambdas = [0.0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
models=[None] * len(lambdas)
for i in range(len(lambdas)):
    lambda_ = lambdas[i]
    models[i] =  Sequential(
        [
            Dense(120, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)),
            Dense(40, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)),
            Dense(classes, activation = 'linear')
        ]
    )
    models[i].compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(0.01),
    )

    models[i].fit(
        X_train,y_train,
        epochs=1000
    )
    print(f"Finished lambda = {lambda_}")


In [ ]:
plot_iterate(lambdas, models, X_train, y_train, X_cv, y_cv)

As regularization is increased, the performance of the model on the training and cross-validation data sets converge. For this data set and model, lambda > 0.01 seems to be a reasonable choice.

<a name="4.5"></a>
### 4.5 Test
Let's try our optimized models on the test set and compare them to 'ideal' performance.

In [ ]:
def recat(pt, origins):
    """ categorize a point based on distance from origin of clusters """
    nclusters = len(origins)
    min_dist = 10000
    y_new = None
    for j in range(nclusters):
        temp = origins[j] - pt.reshape(2,)
        #print(temp.shape,origins[j].shape)
        dist = np.sqrt(np.dot(temp.T, temp))
        if dist < min_dist:
            y_new = j
            min_dist = dist
    return(y_new)

def plt_compare(X,y, classes, simple, regularized, centers):
    plt.close("all")
    fig,ax = plt.subplots(1,3, figsize=(8,3))
    fig.canvas.toolbar_visible = False
    fig.canvas.header_visible = False
    fig.canvas.footer_visible = False

  #plt simple
    plot_cat_decision_boundary(ax[0], X, simple,  vector=True)
    ax[0].set_title("Simple Model", fontsize=14)
    plt_mc_data(ax[0], X,y, classes, map=dkcolors_map, legend=True, size=75)
    ax[0].set_xlabel('x0') ; ax[0].set_ylabel("x1");

  #plt regularized
    plot_cat_decision_boundary(ax[1], X, regularized,  vector=True)
    ax[1].set_title("Regularized Model", fontsize=14)
    plt_mc_data(ax[1], X,y, classes, map=dkcolors_map, legend=True, size=75)
    ax[1].set_xlabel('x0') ; ax[0].set_ylabel("x1");

  #plt ideal
    cat_predict = lambda pt: recat(pt.reshape(1,2), centers)
    plot_cat_decision_boundary(ax[2], X, cat_predict,  vector=False)
    ax[2].set_title("Ideal Model", fontsize=14)
    plt_mc_data(ax[2], X,y, classes, map=dkcolors_map, legend=True, size=75)
    ax[2].set_xlabel('x0') ; ax[0].set_ylabel("x1");

    err_s = eval_cat_err(y, simple(X))
    err_r = eval_cat_err(y, regularized(X))
    ax[0].text(-2.75,3,f"err_test={err_s:0.2f}", fontsize=12)
    ax[1].text(-2.75,3,f"err_test={err_r:0.2f}", fontsize=12)
    m = len(X)
    y_eq  = np.zeros(m)
    for i in range(m):
        y_eq[i] = recat(X[i], centers)
    err_eq = eval_cat_err(y, y_eq)
    ax[2].text(-2.75,3,f"err_test={err_eq:0.2f}", fontsize=12)
    plt.show()

In [ ]:
plt_compare(X_test,y_test, classes, model_predict_s, model_predict_r, centers)

Our test set is small and seems to have a number of outliers so classification error is high. However, the performance of our optimized models is comparable to ideal performance.